# Exercises - JAX/Flax Version

## 0. Setup your own repo
- Don't work in this repo. Set up your own repo to work in. Use `MADS-ML-{yourname}` as a format, eg `MADS-ML-JoostB`.
- You can add `jax`, `flax`, `optax`, `tensorboard`, and `toml` as dependencies.
- Keep `mads_datasets` and `mltrainer` for now (we only use the datasets part).
- Invite me (raoulg; https://github.com/raoulg) as a collaborator to your repo.

# 1. Tune the network
Run the experiment below, explore the different parameters and study the result with tensorboard.
Make a single page (1 A4) report of your findings. Use your visualization skills to communicate your most important findings.

## Key Differences from PyTorch
In JAX/Flax, we make the **P x A → B** paradigm explicit:
- **P** (parameters) are stored separately from the model
- **A** (activation/input) is passed to the model
- **B** (output) is computed via `model.apply(params, input)`

This functional approach makes it clear that parameters are data being threaded through functions.

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
import optax
from flax.training import train_state
from torch.utils.tensorboard import SummaryWriter
from pathlib import Path
from datetime import datetime

## Data Loading

### Option 1: Using mads_datasets (Current)
This keeps compatibility with your existing code.

In [ ]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import BasePreprocessor

# Create dataset factory
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
preprocessor = BasePreprocessor()

# Create data streamers (these are iterators that yield batches)
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
train_streamer = streamers["train"]
valid_streamer = streamers["valid"]

print(f"Train batches: {len(train_streamer)}")
print(f"Valid batches: {len(valid_streamer)}")

### Option 2: Using HuggingFace Datasets (Future Migration)

HuggingFace datasets work great with JAX and avoid TensorFlow dependencies.
Uncomment this section when ready to migrate:

In [ ]:
# # Uncomment to use HuggingFace datasets instead
# from datasets import load_dataset
#
# def create_dataloader_hf(batch_size: int = 64):
#     """Create data loaders using HuggingFace datasets."""
#
#     # Load Fashion MNIST from HuggingFace
#     ds = load_dataset('fashion_mnist')
#
#     class HFDataIterator:
#         """Iterator wrapper for HuggingFace dataset."""
#         def __init__(self, dataset, batch_size, shuffle=True):
#             self.dataset = dataset
#             self.batch_size = batch_size
#             self.shuffle = shuffle
#             self._length = len(dataset) // batch_size
#
#         def __len__(self):
#             return self._length
#
#         def stream(self):
#             """Generator that yields batches (mimics mads_datasets API)."""
#             dataset = self.dataset.shuffle() if self.shuffle else self.dataset
#
#             for i in range(0, len(dataset), self.batch_size):
#                 batch = dataset[i:i + self.batch_size]
#                 # Convert to JAX arrays
#                 images = jnp.array(batch['image'], dtype=jnp.float32).reshape(-1, 28, 28, 1) / 255.0
#                 labels = jnp.array(batch['label'], dtype=jnp.int32)
#                 yield {'image': images, 'label': labels}
#
#     train_streamer = HFDataIterator(ds['train'], batch_size, shuffle=True)
#     valid_streamer = HFDataIterator(ds['test'], batch_size, shuffle=False)
#
#     return train_streamer, valid_streamer
#
# # Uncomment to use:
# # train_streamer, valid_streamer = create_dataloader_hf(batch_size=64)
# # print(f"Train batches: {len(train_streamer)}")
# # print(f"Valid batches: {len(valid_streamer)}")

## Model Definition

In Flax, models are defined as classes that inherit from `nn.Module`. The key difference from PyTorch:
- The model is **stateless** - it doesn't store parameters
- Parameters are initialized separately using `model.init()`
- Forward pass requires explicitly passing parameters via `model.apply(params, x)`

In [ ]:
class NeuralNetwork(nn.Module):
    """Simple MLP for Fashion MNIST classification.

    This demonstrates the P x A → B paradigm:
    - Parameters P are stored separately
    - Input A is passed to __call__
    - Output B is computed via apply(P, A)
    """

    num_classes: int
    units1: int
    units2: int

    @nn.compact
    def __call__(self, x):
        x = x.reshape((x.shape[0], -1))

        x = nn.Dense(features=self.units1)(x)
        x = nn.relu(x)

        x = nn.Dense(features=self.units2)(x)
        x = nn.relu(x)

        x = nn.Dense(features=self.num_classes)(x)

        return x


# Create model (this is just the structure, no parameters yet)
model = NeuralNetwork(num_classes=10, units1=256, units2=256)

# Initialize parameters P
rng = random.PRNGKey(0)
sample_input = jnp.ones((1, 28, 28, 1))  # Dummy input for initialization
params = model.init(rng, sample_input)  # P is here!

print("Model structure:", model)
print("\nParameter shapes:")
print(jax.tree_util.tree_map(lambda x: x.shape, params))

## Understanding P x A → B

Let's demonstrate the explicit parameter passing:

In [ ]:
output = model.apply(params, sample_input)  # P x A → B

print(f"Input shape (A): {sample_input.shape}")
print(f"Output shape (B): {output.shape}")

## Training State

Flax uses `TrainState` to bundle parameters, optimizer state, and training step together.

In [ ]:
def create_train_state(rng, model, learning_rate=1e-3):
    """Create initial training state."""
    sample_input = jnp.ones((1, 28, 28, 1))
    params = model.init(rng, sample_input)

    # Create optimizer
    tx = optax.adam(learning_rate)

    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

## Loss and Metrics

Notice how we explicitly pass parameters to compute loss and metrics.

In [ ]:
def cross_entropy_loss(logits, labels):
    """Compute cross-entropy loss."""
    one_hot_labels = jax.nn.one_hot(labels, num_classes=10)
    return optax.softmax_cross_entropy(logits, one_hot_labels).mean()


def compute_metrics(logits, labels):
    """Compute accuracy."""
    predictions = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(predictions == labels)
    return {"accuracy": accuracy}


@jax.jit
def train_step(state, batch):
    """Single training step."""

    def loss_fn(params):
        # params x image → logits
        logits = state.apply_fn(params, batch["image"])
        loss = cross_entropy_loss(logits, batch["label"])
        return loss, logits

    # Compute gradients
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(state.params)

    # Update parameters P' = P - lr * ∇L(P)
    state = state.apply_gradients(grads=grads)

    # Compute metrics
    metrics = compute_metrics(logits, batch["label"])
    metrics["loss"] = loss

    return state, metrics


@jax.jit
def eval_step(state, batch):
    """Single evaluation step."""
    # P x A → B
    logits = state.apply_fn(state.params, batch["image"])
    loss = cross_entropy_loss(logits, batch["label"])

    metrics = compute_metrics(logits, batch["label"])
    metrics["loss"] = loss

    return metrics

## Training Loop

The training loop shows how parameters are explicitly updated and passed through.

In [ ]:
valid_iter = iter(valid_streamer.stream())
X, y = next(valid_iter)
X.shape, y.shape

In [ ]:
def train_epoch(state, train_streamer, epoch, train_steps, writer):
    """Train for one epoch.

    Args:
        state: Training state containing model and optimizer
        train_streamer: Iterator that yields batches {'image': ..., 'label': ...}
        epoch: Current epoch number
        train_steps: Number of training steps per epoch
        writer: TensorBoard writer
    """
    epoch_metrics = []

    # Get iterator from streamer
    train_iter = iter(train_streamer.stream())

    for step in range(train_steps):
        try:
            X, y = next(train_iter)
            batch = {"image": jnp.array(X), "label": jnp.array(y)}

            # Single training step: P_new, metrics = train_step(P_old, A)
            state, metrics = train_step(state, batch)
            epoch_metrics.append(metrics)

            # Log to tensorboard
            global_step = epoch * train_steps + step
            writer.add_scalar("train/loss", float(metrics["loss"]), global_step)
            writer.add_scalar("train/accuracy", float(metrics["accuracy"]), global_step)

        except StopIteration:
            break

    # Average metrics
    epoch_metrics = jax.tree_util.tree_map(
        lambda *x: jnp.mean(jnp.array(x)), *epoch_metrics
    )

    return state, epoch_metrics


def eval_model(state, valid_streamer, valid_steps):
    """Evaluate the model.

    Args:
        state: Training state
        valid_streamer: Iterator that yields validation batches
        valid_steps: Number of validation steps
    """
    metrics_list = []

    # Get iterator from streamer
    valid_iter = iter(valid_streamer.stream())

    for step in range(valid_steps):
        try:
            X, y = next(valid_iter)
            batch = {"image": jnp.array(X), "label": jnp.array(y)}

            metrics = eval_step(state, batch)
            metrics_list.append(metrics)

        except StopIteration:
            break

    # Average metrics
    metrics = jax.tree_util.tree_map(lambda *x: jnp.mean(jnp.array(x)), *metrics_list)

    return metrics


def train_model(model, config, train_streamer, valid_streamer, logdir):
    """Complete training loop.

    Args:
        model: Flax model
        config: Configuration dictionary
        train_streamer: Training data streamer
        valid_streamer: Validation data streamer
        logdir: Directory for logs
    """
    # Create log directory
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(logdir) / f"run_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Initialize tensorboard writer
    writer = SummaryWriter(str(run_dir))

    # Create initial training state (initialize P)
    rng = random.PRNGKey(config["seed"])
    state = create_train_state(rng, model, config["learning_rate"])

    print("Training configuration:")
    print(f"  Epochs: {config['epochs']}")
    print(f"  Units: [{config['units1']}, {config['units2']}]")
    print(f"  Learning rate: {config['learning_rate']}")
    print(f"  Batch size: {config['batch_size']}")
    print(f"\nLogging to: {run_dir}\n")

    # Training loop
    for epoch in range(config["epochs"]):
        # Train
        state, train_metrics = train_epoch(
            state, train_streamer, epoch, config["train_steps"], writer
        )

        # Evaluate
        valid_metrics = eval_model(state, valid_streamer, config["valid_steps"])

        # Log validation metrics
        writer.add_scalar("valid/loss", float(valid_metrics["loss"]), epoch)
        writer.add_scalar("valid/accuracy", float(valid_metrics["accuracy"]), epoch)

        print(f"Epoch {epoch + 1}/{config['epochs']}:")
        print(
            f"  Train - Loss: {train_metrics['loss']:.4f}, "
            f"Accuracy: {train_metrics['accuracy']:.4f}"
        )
        print(
            f"  Valid - Loss: {valid_metrics['loss']:.4f}, "
            f"Accuracy: {valid_metrics['accuracy']:.4f}"
        )

    writer.close()
    return state

## Single Experiment

Run a single training experiment to verify everything works.

In [ ]:
# Configuration
config = {
    "epochs": 3,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "train_steps": 100,
    "valid_steps": 100,
    "units1": 256,
    "units2": 256,
    "num_classes": 10,
    "seed": 0,
}

# Create model
model = NeuralNetwork(
    num_classes=config["num_classes"], units1=config["units1"], units2=config["units2"]
)

# Train
final_state = train_model(
    model, config, train_streamer, valid_streamer, logdir="modellogs"
)

## Grid Search

Now let's run multiple experiments with different hyperparameters.

In [ ]:
# Grid search over units
units_list = [256, 128, 64]

for units1 in units_list:
    for units2 in units_list:
        print(f"\n{'=' * 60}")
        print(f"Training with units1={units1}, units2={units2}")
        print(f"{'=' * 60}\n")

        config = {
            "epochs": 3,
            "batch_size": 64,
            "learning_rate": 1e-3,
            "train_steps": len(train_streamer),  # Full dataset
            "valid_steps": len(valid_streamer),  # Full validation set
            "units1": units1,
            "units2": units2,
            "num_classes": 10,
            "seed": 0,
        }

        model = NeuralNetwork(
            num_classes=config["num_classes"], units1=units1, units2=units2
        )

        final_state = train_model(
            model, config, train_streamer, valid_streamer, logdir="modellogs"
        )

## Viewing Results

To view results in TensorBoard:

```bash
tensorboard --logdir=modellogs
```

Then open your browser to `localhost:6006`

## Pedagogical Notes on P x A → B

Throughout this notebook, notice how parameters P are always explicit:

1. **Initialization**: `params = model.init(rng, dummy_input)` creates P
2. **Forward pass**: `logits = model.apply(params, input)` computes P x A → B
3. **Gradient computation**: `grads = grad_fn(params)` computes ∂L/∂P
4. **Parameter update**: `state.apply_gradients(grads)` produces P' = P - lr * ∇L(P)

This makes the functional nature of neural networks explicit:
- The model is just a **function** f
- Parameters P **parameterize** that function: f_P
- Training finds P that minimizes loss: P* = argmin_P L(f_P(x), y)

Compare this to PyTorch where parameters are hidden inside `model.parameters()` and updates happen via `optimizer.step()` without explicitly seeing P.

### Bonus: Understanding Flax NNX

Flax NNX is a newer API that looks more stateful (like PyTorch), but you can always extract explicit parameters:

```python
# NNX looks stateful
model = nnx.Linear(2, 3, rngs=nnx.Rngs(0))
output = model(x)  # Where are the parameters?

# But you can extract them!
graphdef, params = nnx.split(model)  # P is here!

# And use them explicitly
model = nnx.merge(graphdef, params)  # P x A → B
```

The stateful API is just syntactic sugar - underneath it's still functional!

# Report Instructions

## 1. Experiment
Experiment with:
- Number of epochs (5, 10, etc.)
- Units in layers: 16, 32, 64, 128, 256, 512, 1024
- Batch size: 4, 8, 16, 32, 64, 128
- Network depth: add more layers to the model
- Learning rate: 1e-2, 1e-3, 1e-4, 1e-5
- Different optimizers from [optax](https://optax.readthedocs.io/): SGD, Adam, RMSprop, etc.

## 2. Study Questions
- **Epochs**: What is the upside/downside of increasing epochs? Do you need more epochs to find the best configuration?
- **Powers of 2**: What is the upside of using factors of 2 for hypertuning? What is a downside?
- **Statefulness**: How does explicit parameter passing in Flax (P x A → B) compare to PyTorch's implicit state? Does it make the learning algorithm clearer?
- **JAX vs PyTorch**: What differences did you notice in training speed? In code clarity?

## 3. Reflect
Follow the scientific method:
- Make a hypothesis
- Design an experiment  
- Run the experiment
- Analyze results and draw conclusions
- Repeat

Keep a journal of your process using [jrnl](https://jrnl.sh/) or similar tool.

## 4. Make a Report
Create a 1 A4 page report covering:
- Your hypothesis about hyperparameter interactions
- What you found
- Visualizations of hyperparameter relationships (e.g., heatmap of units vs accuracy)
- Reflections connecting experimental results to theory
- (Optional) Reflections on the P x A → B paradigm vs PyTorch's implicit parameters